# Flan-T5 Base Model Fine-Tuning for Question-Answering
ref: https://github.com/philschmid/deep-learning-pytorch-huggingface/blob/main/training/flan-t5-samsum-summarization.ipynb

In [ ]:
# install requirements (if needed)
!pip install -r 'requirements.txt'

In [ ]:
import torch
import sys
import os
from subprocess import call
print('_____Python, Pytorch, Cuda info____')
print('__Python VERSION:', sys.version)
print('__pyTorch VERSION:', torch.__version__)
print('__CUDA RUNTIME API VERSION')
#os.system('nvcc --version')
print('__CUDNN VERSION:', torch.backends.cudnn.version())
print('_____nvidia-smi GPU details____')
call(["nvidia-smi", "--format=csv", "--query-gpu=index,name,driver_version,memory.total,memory.used,memory.free"])
print('_____Device assignments____')
print('Number CUDA Devices:', torch.cuda.device_count())
print ('Current cuda device: ', torch.cuda.current_device(), ' **May not correspond to nvidia-smi ID above, check visibility parameter')
print("Device name: ", torch.cuda.get_device_name(torch.cuda.current_device()))

In [1]:
import evaluate
import time
import numpy as np
import torch

import nltk
from nltk.tokenize import sent_tokenize
nltk.download("punkt")

from transformers import (
    AutoModelForSeq2SeqLM
    ,AutoTokenizer
    ,DataCollatorForSeq2Seq
    ,Seq2SeqTrainer
    ,Seq2SeqTrainingArguments
    ,BitsAndBytesConfig
    ,AutoModelForCausalLM
)

from peft import (
    PeftConfig, PeftModel
)

import warnings
warnings.simplefilter("ignore", UserWarning) # ignore user warning about use_reentrant method while fine-tuning

import multiprocessing

# check number of CPU cores
print(f'Currently have {multiprocessing.cpu_count()} CPU cores.')

# check if CUDA GPU is available on torch
print(f'Is CUDA available on torch? {torch.cuda.is_available()}.')

'NoneType' object has no attribute 'cadam32bit_grad_fp32'
Currently have 11 CPU cores.
Is CUDA available on torch? False.


[nltk_data] Downloading package punkt to /Users/dahliama/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
/Users/dahliama/anaconda3/lib/python3.11/site-packages/bitsandbytes/cextension.py:34: UserWarning: The installed version of bitsandbytes was compiled without GPU support. 8-bit optimizers, 8-bit multiplication, and GPU quantization are unavailable.
  warn("The installed version of bitsandbytes was compiled without GPU support. "


In [2]:
# more packages to run falcon model
from peft import LoraConfig, get_peft_model, PeftConfig, PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, AutoTokenizer, GenerationConfig

In [ ]:
!nvidia-smi

## Load pre-trained model & tokenizer

In [166]:
# # specify device type to use to run LLMs (i.e. CPU, GPU/CUDA, etc)
# device_type = "cuda:0"

# load pre-trained model and tokenizer
model_name = "google/flan-t5-base"

# model = AutoModelForSeq2SeqLM.from_pretrained(
#     model_name
#     ,device_map = device_type
#     ,trust_remote_code=True
# )

tokenizer = AutoTokenizer.from_pretrained(model_name ,trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"  # set padding to the right to avoid issues with fp16 (when using 4-bit quantization)

# # check model parameters & structure
# model

## Load & preprocess data

In [200]:
from datasets import load_dataset

In [201]:
data_files = {'train':'train_data.json',
              'validation':'val_data.json',
              'test':'test_data.json'}
data = load_dataset("json", data_files=data_files)
data

  0%|          | 0/3 [00:00<?, ?it/s]

DatasetDict({
    train: Dataset({
        features: ['answer', 'question'],
        num_rows: 17178
    })
    validation: Dataset({
        features: ['answer', 'question'],
        num_rows: 3680
    })
    test: Dataset({
        features: ['answer', 'question'],
        num_rows: 3680
    })
})

Need to preprocess data by truncating and padding before fine-tuning model

In [202]:
from datasets import concatenate_datasets

In [203]:
# The minimum total input sequence length after tokenization. 
# Sequences longer than this will be truncated, sequences shorter will be padded.
tokenized_inputs = concatenate_datasets([data["train"], data["validation"], data["test"]]).map(lambda x: tokenizer(x["question"], truncation=True), batched=True, remove_columns=["question", "answer"])
max_source_length = min([len(x) for x in tokenized_inputs["input_ids"]])
print(f"Min source length: {max_source_length}")

# The minimum total sequence length for target text after tokenization. 
# Sequences longer than this will be truncated, sequences shorter will be padded."
tokenized_targets = concatenate_datasets([data["train"], data["validation"], data["test"]]).map(lambda x: tokenizer(x["answer"], truncation=True), batched=True, remove_columns=["question", "answer"])
max_target_length = min([len(x) for x in tokenized_targets["input_ids"]])
print(f"Min target length: {max_target_length}")

Min source length: 5
Min target length: 2


In [204]:
# The average total input sequence length after tokenization. 
# Sequences longer than this will be truncated, sequences shorter will be padded.
tokenized_inputs = concatenate_datasets([data["train"], data["validation"], data["test"]]).map(lambda x: tokenizer(x["question"], truncation=True), batched=True, remove_columns=["question", "answer"])
max_source_length = np.mean([len(x) for x in tokenized_inputs["input_ids"]])
print(f"Average source length: {max_source_length}")

# The average total sequence length for target text after tokenization. 
# Sequences longer than this will be truncated, sequences shorter will be padded."
tokenized_targets = concatenate_datasets([data["train"], data["validation"], data["test"]]).map(lambda x: tokenizer(x["answer"], truncation=True), batched=True, remove_columns=["question", "answer"])
max_target_length = np.mean([len(x) for x in tokenized_targets["input_ids"]])
print(f"Average target length: {max_target_length}")

Average source length: 14.266566142309888
Average target length: 37.10359442497351


In [205]:
# The maximum total input sequence length after tokenization. 
# Sequences longer than this will be truncated, sequences shorter will be padded.
tokenized_inputs = concatenate_datasets([data["train"], data["validation"], data["test"]]).map(lambda x: tokenizer(x["question"], truncation=True), batched=True, remove_columns=["question", "answer"])
max_source_length = max([len(x) for x in tokenized_inputs["input_ids"]])
print(f"Max source length: {max_source_length}")

# The maximum total sequence length for target text after tokenization. 
# Sequences longer than this will be truncated, sequences shorter will be padded."
tokenized_targets = concatenate_datasets([data["train"], data["validation"], data["test"]]).map(lambda x: tokenizer(x["answer"], truncation=True), batched=True, remove_columns=["question", "answer"])
max_target_length = max([len(x) for x in tokenized_targets["input_ids"]])
print(f"Max target length: {max_target_length}")

Max source length: 47
Max target length: 512


In [206]:
def preprocess_function(sample, padding="max_length"):
    '''
    Purpose: to preprocess data by truncating or padding
    @params sample: data samples
    returns: tokenized dataset that are same lengths
    '''
    # add prefix to the input for t5
    inputs = ["Please answer this question: " + item for item in sample["question"]]

    # tokenize inputs
    model_inputs = tokenizer(inputs, max_length=max_source_length, padding=padding, truncation=True)

    # Tokenize targets with the `text_target` keyword argument
    labels = tokenizer(text_target=sample["answer"], max_length=max_target_length, padding=padding, truncation=True)

    # If we are padding here, replace all tokenizer.pad_token_id in the labels by -100 when we want to ignore
    # padding in the loss.
    if padding == "max_length":
        labels["input_ids"] = [
            [(l if l != tokenizer.pad_token_id else -100) for l in label] for label in labels["input_ids"]
        ]

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_dataset = data.map(preprocess_function, batched=True, remove_columns=["question", "answer"])
print(f"Keys of tokenized dataset: {list(tokenized_dataset['train'].features)}")

Map:   0%|          | 0/3680 [00:00<?, ? examples/s]

Keys of tokenized dataset: ['input_ids', 'attention_mask', 'labels']


## Check model response prior to fine-tuning

In [68]:
# define function to format data to prompt instruction format
def get_model_response(model, question):
    '''
    Purpose: to generate an answer from model for a given question
    @params model: the loaded LLM model
    @params question: a question for model to answer (str)
    returns: an answer (str)
    '''
    input_ids = tokenizer(question, return_tensors="pt").input_ids.to(device_type)
    outputs = model.generate(input_ids
                             ,max_new_tokens=512
                             ,no_repeat_ngram_size=2
                             ,max_time=20  # max computation time to generate response in seconds
                             ,temperature=1
                             ,top_p=0.1
                             )
    response = tokenizer.batch_decode(outputs, skip_special_tokens=True)
    return response[0]

In [111]:
# check response of model response before it is fine-tuned
questions = data['train']['question'][0:5]

for q in questions:
    print(f'Question: {q}')
    q = f'{q}'
    ans = get_model_response(model, q)
    print(f'Answer: {ans}\n')

Question: What is the history of the Soft Coated Wheaten Terrier?
Answer: The Soft Coated Wheaten Terrier was a breed of dog that was first introduced in the United States in 1939. The dog was originally bred as 'Sweet Coat', but was later renamed Swift' in honor of the American dog pioneer, Charles S. Wheaton.

Question: Why are there many flea products available?
Answer: fleas are a common symptom of swine flu.

Question: What are the behavioral reasons for dogs eating cat poop?
Answer: a poop sucks

Question: Why should you continue to reward your dog with treats even after it has reached you?
Answer: It is a reward for good behavior.

Question: Are there other brand names for carprofen?
Answer: no



## Fine-tune model

In [24]:
# set evaluation metric
metric = evaluate.load("rouge")

# helper function to postprocess text
def postprocess_text(preds, labels):
    preds = [pred.strip() for pred in preds]
    labels = [label.strip() for label in labels]

    # rougeLSum expects newline after each sentence
    preds = ["\n".join(sent_tokenize(pred)) for pred in preds]
    labels = ["\n".join(sent_tokenize(label)) for label in labels]

    return preds, labels

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    # Replace -100 in the labels as we can't decode them.
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # Some simple post-processing
    decoded_preds, decoded_labels = postprocess_text(decoded_preds, decoded_labels)

    result = metric.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)
    result = {k: round(v * 100, 4) for k, v in result.items()}
    prediction_lens = [np.count_nonzero(pred != tokenizer.pad_token_id) for pred in preds]
    result["gen_len"] = np.mean(prediction_lens)
    return result

In [25]:
# ignore tokenizer pad token in the loss
label_pad_token_id = -100

# set up data collator to do padding for data inputs and labels
data_collator = DataCollatorForSeq2Seq(
    tokenizer,
    model=model,
    label_pad_token_id=label_pad_token_id,
    pad_to_multiple_of=8
)

In [ ]:
import wandb
wandb.login(key="4a3e504f5d3ee54fa14bc81219ad7cc07a4e0a4c") # log into wandb
# %env WANDB_PROJECT=FLAN-T5-base-paperspace
%env WANDB_PROJECT=FLAN-T5-base-windowsPC

### Experiment 0
All checkpoints reside in "./experiments/google/flan-t5-base_full_finetune"

In [14]:
# define training arguments to fine-tune model
training_args = Seq2SeqTrainingArguments(
    output_dir=f'experiments/{model_name}_full_finetune'
    ,num_train_epochs=10
    ,per_device_train_batch_size=8  # batch size per GPU for training
    ,per_device_eval_batch_size=8
    ,gradient_accumulation_steps=2
    ,gradient_checkpointing=True
    ,optim="paged_adamw_32bit"
    ,logging_steps=10               # log onto console ever 'x' steps
    ,evaluation_strategy="epoch"    # requires an eval_dataset to use
    ,save_strategy="epoch"          # save after every epoch
    ,learning_rate=2e-4
    ,weight_decay=0.001
    ,max_grad_norm=0.3
    ,warmup_ratio=0.03
    # ,lr_scheduler_type="cosine"
    ,disable_tqdm=False
    ,report_to="wandb"
    ,seed=55
    ,predict_with_generate=True
    ,fp16=False
    ,push_to_hub=False
    ,load_best_model_at_end=False
)

# Create Trainer instance
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    compute_metrics=compute_metrics,
)

In [ ]:
# Train/fine tune model
model.config.use_cache = False
trainer.train()

wandb: Currently logged in as: ma-dahlia25. Use `wandb login --relogin` to force relogin
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


You're using a T5TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Epoch,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Rougelsum,Gen Len
1,2.085000,1.908267,39.342000,24.210200,35.657800,36.151800,20.000000
2,1.829700,1.828153,39.655300,24.495100,35.909600,36.457800,20.000000
3,1.690500,1.793843,40.360900,25.203900,36.661500,37.172900,20.000000
4,1.586000,1.777502,40.092300,24.951700,36.382300,36.875700,20.000000
5,1.572800,1.767907,40.644600,25.337200,36.960600,37.447100,20.000000


In [15]:
# Continue to fine-tune model from last checkpoint
model.config.use_cache = False
trainer.train(resume_from_checkpoint=f'experiments/{model_name}_full_finetune/checkpoint-5370')

There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight'].


wandb: Currently logged in as: ma-dahlia25. Use `wandb login --relogin` to force relogin
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


You're using a T5TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Epoch,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Rougelsum,Gen Len
6,1.405000,1.776270,40.572000,25.433700,36.882800,37.367200,20.000000
7,1.373300,1.783993,40.883700,25.604000,37.156400,37.628500,20.000000
8,1.363800,1.790977,40.855300,25.655300,37.049900,37.542500,20.000000
9,1.256400,1.801828,40.906700,25.598900,37.142900,37.640600,20.000000


KeyboardInterrupt: 

### Experiment 1
All checkpoints reside in "./experiments/google/flan-t5-base_full_finetune_1"

In [14]:
# define training arguments to fine-tune model
training_args = Seq2SeqTrainingArguments(
    output_dir=f'experiments/{model_name}_full_finetune_1'
    ,num_train_epochs=8
    ,per_device_train_batch_size=8  # batch size per GPU for training
    ,per_device_eval_batch_size=8
    ,gradient_accumulation_steps=2
    ,gradient_checkpointing=True
    ,optim="paged_adamw_32bit"
    ,logging_steps=10               # log onto console ever 'x' steps
    ,evaluation_strategy="epoch"    # requires an eval_dataset to use
    ,save_strategy="epoch"          # save after every epoch
    ,learning_rate=2e-3
    ,weight_decay=0.001
    ,max_grad_norm=0.3
    ,warmup_ratio=0.03
    # ,lr_scheduler_type="cosine"
    ,disable_tqdm=False
    ,report_to="wandb"
    ,seed=55
    ,predict_with_generate=True
    ,fp16=False
    ,push_to_hub=False
    ,load_best_model_at_end=False
)

# Create Trainer instance
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    compute_metrics=compute_metrics,
)

In [15]:
# Train/fine tune model
model.config.use_cache = False
trainer.train()

wandb: Currently logged in as: ma-dahlia25. Use `wandb login --relogin` to force relogin
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


You're using a T5TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Epoch,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Rougelsum,Gen Len
1,2.192100,2.048818,37.273300,22.237800,33.657900,34.209500,20.000000
2,1.773400,1.928428,39.269900,23.993300,35.500900,36.099700,20.000000
3,1.430900,1.917174,39.607400,24.352900,35.884300,36.427200,20.000000
4,1.112200,1.959772,39.956100,24.889900,36.238700,36.736400,20.000000


KeyboardInterrupt: 

### Experiment 2
All checkpoints reside in "./experiments/google/flan-t5-base_full_finetune_2"

In [14]:
# define training arguments to fine-tune model
training_args = Seq2SeqTrainingArguments(
    output_dir=f'experiments/{model_name}_full_finetune_2'
    ,num_train_epochs=5
    ,per_device_train_batch_size=8  # batch size per GPU for training
    ,per_device_eval_batch_size=8
    ,gradient_accumulation_steps=2
    ,gradient_checkpointing=True
    ,optim="paged_adamw_32bit"
    ,logging_steps=10               # log onto console ever 'x' steps
    ,evaluation_strategy="epoch"    # requires an eval_dataset to use
    ,save_strategy="epoch"          # save after every epoch
    ,learning_rate=2e-3
    ,weight_decay=0.01
    ,max_grad_norm=0.3
    ,warmup_ratio=0.03
    # ,lr_scheduler_type="cosine"
    ,disable_tqdm=False
    ,report_to="wandb"
    ,seed=55
    ,predict_with_generate=True
    ,fp16=False
    ,push_to_hub=False
    ,load_best_model_at_end=False
)

# Create Trainer instance
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    compute_metrics=compute_metrics,
)

In [15]:
# Train/fine tune model
model.config.use_cache = False
trainer.train()

wandb: Currently logged in as: ma-dahlia25. Use `wandb login --relogin` to force relogin
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


You're using a T5TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Epoch,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Rougelsum,Gen Len
1,2.155600,2.020701,37.935800,22.920000,34.487200,34.997000,20.000000
2,1.694400,1.892979,39.330400,24.204400,35.668500,36.208700,20.000000
3,1.304400,1.887984,39.797800,24.732900,36.086000,36.586600,20.000000
4,0.986600,1.948059,40.444700,25.512400,36.784600,37.247600,20.000000
5,0.758500,2.105859,40.799200,25.908900,37.102700,37.593800,20.000000


TrainOutput(global_step=5370, training_loss=1.4672883983874898, metrics={'train_runtime': 16347.6018, 'train_samples_per_second': 5.254, 'train_steps_per_second': 0.328, 'total_flos': 5513794593914880.0, 'train_loss': 1.4672883983874898, 'epoch': 5.0})

### Experiment 3
All checkpoints reside in "./experiments/google/flan-t5-base_full_finetune_3"

In [14]:
# define training arguments to fine-tune model
training_args = Seq2SeqTrainingArguments(
    output_dir=f'experiments/{model_name}_full_finetune_3'
    ,num_train_epochs=5
    ,per_device_train_batch_size=8  # batch size per GPU for training
    ,per_device_eval_batch_size=8
    ,gradient_accumulation_steps=2
    ,gradient_checkpointing=True
    ,optim="paged_adamw_32bit"
    ,logging_steps=10               # log onto console ever 'x' steps
    ,evaluation_strategy="epoch"    # requires an eval_dataset to use
    ,save_strategy="epoch"          # save after every epoch
    ,learning_rate=2e-5
    ,weight_decay=0.001
    ,max_grad_norm=0.3
    ,warmup_ratio=0.03
    # ,lr_scheduler_type="cosine"
    ,disable_tqdm=False
    ,report_to="wandb"
    ,seed=55
    ,predict_with_generate=True
    ,fp16=False
    ,push_to_hub=False
    ,load_best_model_at_end=False
)

# Create Trainer instance
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    compute_metrics=compute_metrics,
)

In [15]:
# Train/fine tune model
model.config.use_cache = False
trainer.train()

wandb: Currently logged in as: ma-dahlia25. Use `wandb login --relogin` to force relogin
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


You're using a T5TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Epoch,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Rougelsum,Gen Len
1,2.263700,2.063978,37.917500,22.650700,34.017900,34.511200,20.000000
2,2.122700,2.009982,38.326000,23.040900,34.522900,35.008700,20.000000
3,2.119300,1.986231,38.770300,23.468700,34.958800,35.481600,20.000000
4,2.113300,1.972360,38.706300,23.477400,34.967400,35.469300,19.999728
5,2.208400,1.968701,38.760000,23.544400,35.020000,35.523200,20.000000


TrainOutput(global_step=5370, training_loss=2.2259293486952116, metrics={'train_runtime': 10306.2208, 'train_samples_per_second': 8.334, 'train_steps_per_second': 0.521, 'total_flos': 5513794593914880.0, 'train_loss': 2.2259293486952116, 'epoch': 5.0})

In [16]:
wandb.finish()

eval/gen_len,███▁█
eval/loss,█▄▂▁▁
eval/rouge1,▁▄█▇█
eval/rouge2,▁▄▇▇█
eval/rougeL,▁▅███
eval/rougeLsum,▁▄███
eval/runtime,█▄▃▁▅
eval/samples_per_second,▁▅▆█▄
eval/steps_per_second,▁▅▆█▄
train/epoch,▁▁▁▂▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇███
train/global_step,▁▁▁▂▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇███


## Run inference & evaluate model

In [6]:
from transformers import pipeline
import evaluate
from tqdm import tqdm
from rouge_score import rouge_scorer

In [66]:
# specify device type to use to run LLMs
device_type = "mps"  # use mps for MacOS

# load fine-tuned model and tokenizer
base_model_name = "google/flan-t5-base"
model_name = "dahlia25/flan-t5-base-dog-full"

trained_model = AutoModelForSeq2SeqLM.from_pretrained(
    model_name
    ,device_map = device_type
    ,trust_remote_code=True
)

tokenizer = AutoTokenizer.from_pretrained(base_model_name ,trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"  # set padding to the right to avoid issues with fp16 (when using 4-bit quantization)

### Evalute model with ROUGE & BLEU metrics

In [208]:
def calculate_rouge4(pred, ref):
    '''
    Purpose: to calculate ROUGE-4 score for a single data point
    @params pred: a list of predictions made by model (str)
    @params ref: a list of references/actual answers (str)
    returns: a dict of evaluation results (dict)
    '''
    # get rouge-4 score; since not included in evaluate load method
    scorer = rouge_scorer.RougeScorer(['rouge4'], use_stemmer=True)
    scores = scorer.score(pred, ref)

    res = scores['rouge4'][2]  # get fmeasure (precision+recall) results
    
    return res

In [109]:
def calculate_rouge_bleu(preds, refs):
    '''
    Purpose: to calculate ROUGE & BLEU metrics for a set of data samples
    @params preds: a list of predictions made by model (list of str)
    @params refs: a list of references/actual answers (list of str)
    returns: a dict of evaluation results (dict)
    '''

    rouge_metric = evaluate.load("rouge")
    rouge_res = rouge_metric.compute(predictions=preds, references=refs, use_stemmer=True)
    
    # calculate average rouge-4 score
    rouge4_res = []
    for i in range(len(preds)):
        pred = preds[i]
        ref = refs[i]
        r4 = calculate_rouge4(pred, ref)
        rouge4_res.append(r4)
        
    r4_avg = np.mean(rouge4_res)
    
    # put all results together
    res = rouge_res
    rouge_res['rouge4'] = r4_avg
    
    # calculate bleu scores
    bleu_metric = evaluate.load("bleu")
    bleu_res = bleu_metric.compute(predictions=preds, references=refs)
    
    for i in range(len(bleu_res['precisions'])):
        name_ix = i+1
        
        k = f'bleu{name_ix}'
        res[k] = bleu_res['precisions'][i]
        
    return res

In [110]:
questions = data['test']['question']
refs = data['test']['answer']  # actual answer

# get model predictions
preds = []
for i in tqdm(range(len(questions))):
    q = questions[i]
    ans = get_model_response(trained_model, q)
    preds.append(ans)

# calculate rouge & bleu metrics
eval_results = calculate_rouge_bleu(preds, refs)

print(f'----------\n Results: \n {eval_results}')

 28%|█████████▍                        | 1021/3680 [4:28:08<11:38:19, 15.76s/it]


KeyboardInterrupt: 

In [114]:
# save generated predictions
with open("preds.txt", "w") as output:
    output.write(str(preds))
output.close()

In [211]:
# manually invoke rouge/bleu calculation if for loop earlier didn't finish
refs = data['test']['answer'][0:len(preds)]  # actual answer
calculate_rouge_bleu(preds, refs)

{'rouge1': 0.21859293615879244,
 'rouge2': 0.09883003031969073,
 'rougeL': 0.16861180081557692,
 'rougeLsum': 0.1685901151885983,
 'rouge4': 0.04082523094137193,
 'bleu1': 0.12945500672453844,
 'bleu2': 0.05437335859461041,
 'bleu3': 0.029901724860660447,
 'bleu4': 0.02010938881542398}

### Evaluate model with Jaro-Winkler similarity metric

In [118]:
from textdistance import jaro_winkler

In [119]:
# calculate jaro-winkler distance
refs = data['test']['answer'][0:len(preds)]  # actual answer

jaros = []
for i in range(len(preds)):
    pred = preds[i]
    ref = refs[i]
    distance = textdistance.jaro_winkler(pred, ref)
    jaros.append(distance)
    
print(f'The average jaro-winkler similarity distance is: {np.mean(jaros)}')

The average jaro-winkler similarity distance is: 0.61583174036946


### Evaluate model with Cosine Embedding distance metric

In [181]:
from langchain.evaluation import load_evaluator
import os

os.environ['OPENAI_API_KEY'] = '<API_KEY>'

In [182]:
# evaluator = load_evaluator("embedding_distance")
evaluator = load_evaluator(
    "embedding_distance", distance_metric=EmbeddingDistance.COSINE
)

In [191]:
# calculate cosine embedding distance
refs = data['test']['answer'][0:len(preds)]  # actual answer

cos_dist = []
for i in tqdm(range(len(preds))):
    pred = preds[i]
    ref = refs[i]
    distance = evaluator.evaluate_strings(prediction=pred, reference=ref)
    cos_dist.append(distance['score'])
    
print(f'The average cosine embedding distance is: {np.mean(cos_dist)}')

100%|███████████████████████████████████████| 1021/1021 [04:18<00:00,  3.95it/s]

The average cosine embedding distance is: 0.1028848957382456


### Run inference to check model output

In [ ]:
# specify device type to use to run LLMs (i.e. CPU, GPU/CUDA, etc)
device_type = "mps"  # use mps for MacOS

# load fine-tuned model and tokenizer
base_model_name = "google/flan-t5-base"
model_name = "dahlia25/flan-t5-base-dog-full"

trained_model = AutoModelForSeq2SeqLM.from_pretrained(
    model_name
    ,device_map = device_type
    ,trust_remote_code=True
)

tokenizer = AutoTokenizer.from_pretrained(base_model_name ,trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"  # set padding to the right to avoid issues with fp16 (when using 4-bit quantization)

# create pipeline
pipe = pipeline(task='text2text-generation'  # not 'question-answer' for T5 models
                ,model=trained_model
                ,tokenizer=tokenizer)

In [17]:
data['train'][41]

{'answer': 'Diagnostic tests for kennel cough may include chest X-rays, routine laboratory blood tests, fecal flotation, and examination of the cornea of the eye.',
 'question': 'What diagnostic tests are needed to recognize kennel cough?'}

In [28]:
sample = data['train'][41]['question']
print(f"question: \n{sample}\n---------------")

# summarize dialogue
res = pipe(f'Respond to this professionally: {sample}'
           ,max_time=5          # max time for computation in seconds (20)
           ,max_new_tokens=512   # max token length to generate
           ,no_repeat_ngram_size=2
           ,temperature=0.1
           ,do_sample=True
           ,early_stopping=True)

print(f"flan-t5-base answer:\n{res}")

question: 
What diagnostic tests are needed to recognize kennel cough?
---------------
flan-t5-base answer:
[{'generated_text': 'Diagnostic tests may include a complete medical history and physical examination, complete blood count, biochemical profile, urinalysis, chest X-rays, and fecal examination. Additional tests such as x-moscopy, radiographs of the chest, abdominal radiography, or bronchoscopy may be recommended. A complete veterinary history, physical exam, blood tests for infectious diseases, electrocardiogram (ECG), and chest radiographic (X) scans may also be performed. Blood tests and radiograms are also recommended for diagnosis. Other tests that may need to be done include complete physical'}]


## Create chatbot UI

In [3]:
# create chatbot interface
from transformers import pipeline
import gradio as gr
import re

In [4]:
def flanT5_predict(pipe, message):
    '''
    Purpose: to use fine-tuned Flan-T5 model to generate response
    @params pipe: the transformers pipe object to generate predictions
    @params message: user input/question (str)
    returns: a response/answer (str)
    '''
    ans = pipe(f'Respond to this professionally: {message}'
               ,max_time=5          # max time for computation in seconds (20)
               ,max_new_tokens=512   # max token length to generate
               ,no_repeat_ngram_size=2
               ,temperature=0.1
               ,do_sample=True
               ,early_stopping=True)
    
    return ans[0]['generated_text']

In [18]:
def falcon_predict(pipe, message):
    '''
    Purpose: to use fine-tuned Falcon 7B model to generate response
    @params pipe: the transformers pipe object to generate predictions
    @params message: user input/question (str)
    returns: a response/answer (str)
    '''   
    ans = pipe(f"{message} Please provide a concise answer."
               ,max_time=10
               ,max_new_tokens=1024
               ,min_length=128
               ,temperature=0
               ,top_k=1
               ,early_stopping=True)
    
    return ans[0]['generated_text']

In [6]:
def truncate_text(text):
    '''
    Purpose: to truncate and remove incomplete sentences
    @params text: text to truncate (str)
    returns: truncated text (str)
    '''
    # Split the text into individual sentences.
    sentences = re.split('(?<=[.?!])\s+', text)

    # Container for the truncated response.
    keep_sentences = []
    for sentence in sentences:
        if sentence.endswith('.'):  # determine if this sentence ends with a period; it is likely the end of the response section.
            keep_sentences.append(sentence)
        else:
            pass
        
    if len(keep_sentences) == 0:
        return sentences[0]
    else:
        return ' '.join(keep_sentences)  # join the processed sentences back into a coherent string

In [11]:
### for FLAN-T5 full ###
base_model_name = "google/flan-t5-base"

flan_tokenizer = AutoTokenizer.from_pretrained(base_model_name, trust_remote_code=True)
flan_tokenizer.pad_token = flan_tokenizer.eos_token
flan_tokenizer.padding_side = "right"  # set padding to the right to avoid issues with fp16 (esp. when using 4-bit quantization)

# set up LLM pipeline with fine-tuned model
flan_model = "dahlia25/flan-t5-base-dog-full"

flan_t5_pipe = pipeline(task='text2text-generation'  # not 'question-answer' for T5 models
                        ,model=flan_model
                        ,tokenizer=flan_tokenizer)

In [4]:
### for Falcon-7B QLoRA ###
base_model_name = "deepaknh/falcon7B_FineTuning_Experiment2_QLORA_7perParam"

config = PeftConfig.from_pretrained(base_model_name)
falcon_tokenizer = AutoTokenizer.from_pretrained(config.base_model_name_or_path)
falcon_tokenizer.pad_token = falcon_tokenizer.eos_token

# set up LLM pipeline with fine-tuned model
falcon_model = "deepaknh/falcon7B_FineTuning_Experiment2_QLORA_7perParam"

falcon_pipe = pipeline(task='text-generation'
                       ,model=falcon_model
                       ,tokenizer=falcon_tokenizer
                       ,eos_token_id=falcon_tokenizer.eos_token_id)

Loading checkpoint shards:   0%|          | 0/15 [00:00<?, ?it/s]

In [ ]:
# testing out falcon code part 1
base_model_name = "deepaknh/falcon7B_FineTuning_Experiment2_QLORA_7perParam"
falcon_model = AutoModelForCausalLM.from_pretrained(base_model_name, trust_remote_code=True)

Loading checkpoint shards:   0%|          | 0/15 [00:00<?, ?it/s]

In [20]:
# testing out falcon code part 2
question = "Is chocolate toxic to dogs?"

input_ids = falcon_tokenizer.encode(question + " Please provide a concise answer.", return_tensors="pt")
eos_token_id = falcon_tokenizer.eos_token_id

outputs = falcon_model.generate(
    input_ids=input_ids,
    max_new_tokens=1024,
    min_length=128,
    eos_token_id=eos_token_id,
    early_stopping=True,
    max_time=10,
    temperature=0,
    top_k=1
)

generated_responses = falcon_tokenizer.batch_decode(outputs, skip_special_tokens=True)
generated_responses

NameError: name 'peft_model' is not defined

In [19]:
message = "Is chocolate toxic to dogs?"
output = falcon_predict(falcon_pipe, message)
output

Setting `pad_token_id` to `eos_token_id`:11 for open-end generation.


'Is chocolate toxic to dogs? Please provide a concise answer.\n'

In [17]:
def predict(message, bot):
    '''
    Purpose: to generate a response for chatbot interface
    @params message: user input/prompt (str)
    @params bot: the model that the users selects as the bot assistant in the chatbot UI (str)
    '''
    ans = flanT5_predict(pipe, message)
    ans = truncate_text(ans)
    return ans

with gr.Blocks() as demo:
    gr.HTML("""Welcome to Project Marley: a DogCare Conversational AI""")
    gr.Markdown(
        """This is a chatbot that will answer any diet and health questions you have regarding to dogs. Save time and money by getting your curiosities answered here before visiting a vet."""
    )
    with gr.Row():
        model_select = gr.Dropdown(["Flan-T5", "Falcon-7B"], label="Select a bot assistant")
    
    with gr.Row():
        with gr.Column():
            gr.ChatInterface(predict)

demo.launch(share=True)

Running on local URL:  http://127.0.0.1:7864


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Running on public URL: https://f7b4aabe2633c10e53.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from Terminal to deploy to Spaces (https://huggingface.co/spaces)
